# Recipe 01: The Bell State

**Companion notebook for [The Quokka Cookbook](https://github.com/johnazariah/quokka-cookbook)**

This notebook loads the Bell state QASM circuit, runs it on a cloud Quokka, and lets you experiment with all four Bell states.

**Requirements:** `requests` (included with most Python installations)

## Connect to Quokka

The cloud Quokkas are at `quokka1-6.quokkacomputing.com`. We just need an HTTP POST — no special SDK required.

In [ ]:
import json
from pathlib import Path

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# Pick a cloud Quokka (1-6). If one is offline, try another.
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    """Send a QASM program to the cloud Quokka and return results."""
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

def run_qasm_file(filepath: str, shots: int = 1024) -> dict:
    """Load a .qasm file and run it."""
    return run_qasm(Path(filepath).read_text(), shots=shots)

print(f"Ready to send circuits to {QUOKKA_URL}")

## Run the Bell State circuit

Load `bell.qasm` from the recipes folder and run it on the cloud Quokka.

Expected result: only `00` and `11` — never `01` or `10`.

In [ ]:
# Load and display the QASM file
bell_qasm = Path("../recipes/01-bell-state/bell.qasm").read_text()
print(bell_qasm)

In [ ]:
# Run it!
results = run_qasm(bell_qasm, shots=1024)
print("Results:", results)

## Visualise the results

In [ ]:
import matplotlib.pyplot as plt

def plot_results(results: dict, title: str = "Measurement outcomes"):
    """Bar chart of measurement outcomes."""
    labels = sorted(results.keys())
    values = [results[k] for k in labels]
    plt.bar(labels, values, color="#009688")
    plt.xlabel("Outcome")
    plt.ylabel("Counts")
    plt.title(title)
    plt.show()

plot_results(results, "Bell State |Φ⁺⟩")

## Experiment: All four Bell states

There are four Bell states, each produced by a small variation of the circuit:

| State | Circuit | Expected outcomes |
|-------|---------|-------------------|
| $|\Phi^+\rangle$ | H, CNOT | `00` + `11` |
| $|\Phi^-\rangle$ | X, H, CNOT | `00` + `11` (but with a relative phase) |
| $|\Psi^+\rangle$ | H, CNOT, X on target | `01` + `10` |
| $|\Psi^-\rangle$ | X, H, CNOT, X on target | `01` + `10` |

In [ ]:
bell_states = {
    "|Φ⁺⟩": """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
""",
    "|Φ⁻⟩": """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
x q[0];
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
""",
    "|Ψ⁺⟩": """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
x q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
""",
    "|Ψ⁻⟩": """
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
x q[0];
h q[0];
cx q[0], q[1];
x q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (name, circuit) in zip(axes, bell_states.items()):
    result = run_qasm(circuit, shots=1024)
    labels = sorted(result.keys())
    values = [result[k] for k in labels]
    ax.bar(labels, values, color="#009688")
    ax.set_title(name, fontsize=14)
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Counts")

plt.suptitle("The Four Bell States — run on Quokka", fontsize=16)
plt.tight_layout()
plt.show()

## What's next?

- Read the [full recipe](../recipes/01-bell-state/README.md) for the theory behind the Bell state
- Try [Recipe 02: Teleportation](../recipes/02-teleportation/README.md) — it uses Bell states to transmit quantum information
- Try [Recipe 03: Deutsch-Jozsa](../recipes/03-deutsch-jozsa/README.md) — your first quantum speedup